# Deep learning on images

## Load modules from repo

In [94]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [95]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [96]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model, get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

In [97]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)

<module 'src.models.on_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

In [9]:
# Class distribution
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [10]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = False  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

try_loading_model=False

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if rebalance_with_weights:
    if small_train_sample:
        raise ValueError("Warning: rebalance_with_weights should not be used with small_train_sample.")
    print('using class weights')
else:
    print('not using class weights')

not using class weights


In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(6793, 31)


In [17]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [ ]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
create_model=False
if try_loading_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le champion : {champion_path}")
        model = keras.models.try_loading_model(champion_path)
        subversion = last_experiment.get('subversion', 1)
    else:
        create_model=True
else:
    create_model=True

if create_model:
    subversion = last_experiment.get('subversion', 0) + 1
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Création d'un nouveau modèle.


### Summary

In [24]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 500, 500,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 500, 500,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 500, 500,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 250, 250,  │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 250, 250,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 250, 250,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 250, 250,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 250, 250,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 250, 250,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 125, 125,  │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 125, 125,  │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 125, 125,  │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 125, 125,  │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 125, 125,  │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 125, 125,  │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 125, 125,  │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 125, 125,  │          0 │ block2b_expand_b

 Total params: 8,164,299 (31.14 MB)

 Trainable params: 2,244,987 (8.56 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

## Callbacks

### ModelCheckpoint

In [ ]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [ ]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [ ]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [ ]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [ ]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [ ]:
tensor_board_folder = artifacts_folder / "tensor_board"
tensor_board = callbacks.TensorBoard(log_dir = tensor_board_folder)

## Configuration 2

### max_epochs

In [ ]:
# max_epochs=4  # TODO: try 50

In [ ]:
# Pick max_epochs based on your available time
available_minutes=10

import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793
max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

4

### misc

In [ ]:
learning_rate=0.001

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [38]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=10, max_epochs=4, champion_path=None ?

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=total_epochs_trained, callbacks=callbacks, verbose=1)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/4


2025-10-02 17:40:35.132478: I external/local_xla/xla/service/service.cc:163] XLA service 0x640c4ee5cfb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-02 17:40:35.132500: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-02 17:40:35.613039: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-02 17:40:37.071158: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-02 17:40:38.068724: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-02 17:40:38.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.2010 - loss: 3.3465

2025-10-02 17:41:45.388577: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:41:45.483374: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:41:46.079743: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:41:46.179113: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:41:46.897528: E external/local_xla/xla/stream_

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.2014 - loss: 3.3449

2025-10-02 17:43:05.447331: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-02 17:43:13.469773: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:43:13.567601: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 17:43:14.395249: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 173s 658ms/step - accuracy: 0.2821 - loss: 3.0155 - val_accuracy: 0.4525 - val_loss: 2.3335 - learning_rate: 0.0010
Epoch 2/4
213/213 ━━━━━━━━━━━━━━━━━━━━ 103s 485ms/step - accuracy: 0.4390 - loss: 2.3639 - val_accuracy: 0.5098 - val_loss: 2.0680 - learning_rate: 0.0010
Epoch 3/4
213/213 ━━━━━━━━━━━━━━━━━━━━ 101s 474ms/step - accuracy: 0.4942 - loss: 2.1150 - val_accuracy: 0.5413 - val_loss: 1.9439 - learning_rate: 0.0010
Epoch 4/4
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 482ms/step - accuracy: 0.5350 - loss: 1.9617 - val_accuracy: 0.5560 - val_loss: 1.9015 - learning_rate: 0.0010


7.9944052735964455

## Evaluation

In [42]:
# To use tensorboard:
# open a terminal from the root of the repo and run these two commands (the path must match the value of tensor_board_folder).
# source venv/bin/activate
# tensorboard --logdir artifacts/on_images/deep_learning/v1/tensor_board
tensor_board_folder

PosixPath('artifacts/on_images/deep_learning/v1/tensor_board')

In [43]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5559938549995422,
 'actual_epochs': 4,
 'minutes_per_epoch': 1.9986013183991114}

In [44]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [45]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 73s 128ms/step


array([[1.3510230e-03, 3.7801932e-03, 6.5643881e-03, ..., 1.6926404e-02,
        1.0205805e-03, 6.5136235e-04],
       [1.9625667e-03, 1.3192000e-02, 1.5311360e-02, ..., 2.9356385e-02,
        2.2646987e-03, 7.9990719e-03],
       [1.0837539e-04, 3.7831184e-03, 2.6045416e-02, ..., 2.1016152e-01,
        9.7588418e-05, 4.2317712e-04],
       ...,
       [2.2451985e-01, 2.0579690e-02, 4.2169835e-04, ..., 3.2735022e-04,
        3.5413656e-01, 7.3784390e-03],
       [2.7579616e-03, 8.5899923e-03, 7.9926690e-03, ..., 2.5495384e-02,
        2.6462167e-03, 1.1431646e-03],
       [2.7850573e-03, 6.0540508e-03, 1.2272403e-02, ..., 7.7912606e-02,
        2.7506854e-03, 9.0758811e-04]], shape=(16984, 27), dtype=float32)

In [46]:
from sklearn import metrics

In [47]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [48]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1280,1281,1300,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,
10,169,27,0,0,3,26,5,0,2,0,5,0,1,0,8,0,164,138,3,4,0,4,0,64,0
40,6,233,3,4,7,49,10,1,47,5,1,3,1,0,8,0,44,16,25,4,1,5,5,21,3
50,1,13,48,37,13,7,17,0,82,5,10,3,0,0,11,0,2,8,26,6,2,43,2,0,0
60,0,8,6,110,0,4,1,0,11,0,0,0,0,0,2,0,2,0,17,0,0,3,0,0,2
1140,1,19,3,1,287,30,106,0,10,3,16,0,2,0,6,0,7,5,14,0,0,16,1,5,2
1160,0,36,0,0,8,697,2,0,0,0,0,0,0,0,1,0,22,13,9,1,0,1,0,1,0
1180,0,4,1,0,39,19,23,0,4,0,4,1,0,0,4,0,11,18,6,6,1,8,0,3,1
1280,1,21,6,5,60,9,442,6,193,25,23,14,10,0,62,0,6,15,6,9,4,44,8,3,2
1281,6,33,3,5,8,47,107,17,15,10,6,2,1,2,23,0,24,36,14,10,1,23,1,13,7


In [59]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.5249893344543362, 0.0)

In [50]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [51]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

,precision,recall,f1-score,support
10,0.528125,0.271268,0.358431,623.000000
40,0.437970,0.464143,0.450677,502.000000
50,0.377953,0.142857,0.207343,336.000000
60,0.533981,0.662651,0.591398,166.000000
1140,0.579798,0.537453,0.557823,534.000000
1160,0.689416,0.881163,0.773585,791.000000
1180,0.000000,0.000000,0.000000,153.000000
1280,0.350515,0.453799,0.395526,974.000000
1281,0.629630,0.041063,0.077098,414.000000
1300,0.518762,0.780971,0.623418,1009.000000


In [52]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.520031,0.449751,0.438975,629.037037
std,0.225780,0.294038,0.247325,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.406335,0.180124,0.275164,310.000000
50%,0.533981,0.464789,0.450677,534.000000
75%,0.657820,0.715383,0.620667,953.500000
max,1.000000,0.881163,0.784211,2042.000000


In [53]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.412213,0.489183,0.031235
recall,0.412213,1.000000,0.975693,0.095890
f1-score,0.489183,0.975693,1.000000,0.090313
support,0.031235,0.095890,0.090313,1.000000


In [54]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.5249893344543362

In [55]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5559938549995422,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5249893344543362,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2473254562505832)}

## Update tracker

In [ ]:
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True

    subversion=last_experiment.get('subversion',1)
    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epochs-{total_epochs_trained + len(model_history.epoch)}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.


In [67]:
tracker['total_epochs'] = total_epochs_trained + len(model_history.epoch)

In [68]:
to_track=['subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5559938549995422,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5249893344543362,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2473254562505832),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/model_subv1_epochs-4_f1-0.5250.keras',
 'total_epochs': 4,
 'subversion': 1,
 'max_epochs': 4,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [69]:
tracker['comment']="Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers."

In [70]:
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5559938549995422,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5249893344543362,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2473254562505832),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/model_subv1_epochs-4_f1-0.5250.keras',
 'total_epochs': 4,
 'subversion': 1,
 'max_epochs': 4,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16},
 'comment': 'Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.'}

In [71]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [72]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [73]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [98]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, log_file_path=log_file_path)

Log pour l'expérience version 1 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet.


## Show tracking logs

In [99]:
pd.set_option('max_colwidth', None)

In [131]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment,best_model_path
0,1,False,20380,32,2.657786,8,13.0,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.,None
1,2,False,6793,32,1.998601,4,4.0,0.001,0.555994,0.524989,0.000000,0.247325,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/model_subv1_epochs-4_f1-0.5250.keras
